Import Libraries

In [2]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn


Load data

In [3]:
df = pd.read_csv("C:\Machine_Learning_Projects\energy-load-forecasting\smart-household-energy-forecasting\data\processed\data.csv")

Feature extraction

In [5]:
data = df['Global_active_power'].values
data

array([4.216, 5.36 , 5.374, ..., 0.938, 0.934, 0.932])

Convert into 2D array

In [6]:
data = data.reshape(-1, 1)
data

array([[4.216],
       [5.36 ],
       [5.374],
       ...,
       [0.938],
       [0.934],
       [0.932]])

Normalize between 0 & 1

In [7]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
data = scaler.fit_transform(data)

Create sequence

In [11]:
import numpy as np

def create_sequences(data, window_size = 60):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i: i + window_size])
        y.append(data[i + window_size])
    
    return np.array(X), np.array(y)

X, y = create_sequences(data, 60) 

Split values for train and test

In [ ]:
split = int(0.8 * len(X))

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

Define LSTM model

In [22]:
class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size = 1, hidden_size = 50, batch_first = True)
        self.fc = nn.Linear(50, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.fc(out)

Convert array into tensor

In [ ]:
X_train = torch.tensor(X_train, dtype = torch.float32)
y_train = torch.tensor(y_train, dtype = torch.float32)

Train data

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

# 1. Wrap your data in a Dataset and Loader
# Ensure X_train and y_train are already Tensors
train_data = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

model = LSTMModel()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 2. Updated Training Loop
epochs = 5
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    # This inner loop pulls only 32 windows at a time
    for batch_X, batch_y in train_loader:
        # Forward pass
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch + 1}, Average Loss: {avg_loss:.6f}")